## Homework 3

> [!NOTE]
> This homework uses the pinned 2026 lead-scoring release in the course
> repository. The plan and report are available in `cohorts/2026/data/`.

### Dataset

In this homework, we will use the 2026 lead-scoring dataset. Download it from [here](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv).

Or you can do it with `wget`:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
```

In this dataset our desired target for classification task will be `converted` variable - has the client signed up to the platform or not.

In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score
from sklearn.model_selection import train_test_split


In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
csv_file = '../assets/course_lead_scoring_2026.csv'
!wget -qc -O {csv_file} {url}


### Data preparation

* Check if the missing values are presented in the features.
* If there are missing values:
    * For categorical features, replace them with 'NA'
    * For numerical features, replace with with 0.0

In [3]:
df = pd.read_csv(csv_file)
df.head()


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88_160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72_688.0,3,7,0.72,1
2,referral,retail,employed,europe,44_697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24_062.0,4,5,0.62,1


In [4]:
pd.DataFrame({
    'num_missing': df.isna().sum(),
    'dtype': df.dtypes,
    'num_unique_values': df.nunique(dropna=False),
    'unique_values': df.apply(lambda x: x.unique() if x.nunique() <= 20 else '...'),
})

df.fillna(
    dict.fromkeys(df.select_dtypes('string'), 'NA') | dict.fromkeys(df.select_dtypes('number'), 0),
    inplace=True,
).head()

df.isna().sum()

,num_missing,dtype,num_unique_values,unique_values
lead_source,148,str,6,"[organic_search, social_media, referral, paid_ads, nan, events]"
industry,240,str,7,"[technology, retail, manufacturing, education, healthcare, nan, finance]"
employment_status,194,str,5,"[employed, student, nan, unemployed, self_employed]"
location,208,str,6,"[europe, south_america, nan, north_america, africa, asia]"
annual_income,369,float64,4314,...
number_of_courses_viewed,0,int64,7,"[3, 4, 2, 1, 0, 5, 6]"
interaction_count,0,int64,14,"[4, 7, 6, 5, 9, 2, 0, 1, 3, 10, 8, 11, 12, 13]"
lead_score,35,float64,100,...
converted,0,int64,2,"[1, 0]"


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88_160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72_688.0,3,7,0.72,1
2,referral,retail,employed,europe,44_697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24_062.0,4,5,0.62,1


lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
Length: 9, dtype: int64

### Question 1

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology` ✅ 
- `healthcare`
- `retail`

In [5]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, Length: 7, dtype: int64

### Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset.
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score` ✅ 
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

In [6]:
TARGET_VAR = 'converted'

corr_matrix = df.drop(columns=TARGET_VAR).corr(numeric_only=True).round(4)
corr_matrix

features = (
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
)

pd.Series([corr_matrix.loc[x, y] for x, y in features], index=features).sort_values(key=abs, ascending=False)


,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.0,0.1613,0.1228,0.2295
number_of_courses_viewed,0.1613,1.0,0.7216,0.7572
interaction_count,0.1228,0.7216,1.0,0.9157
lead_score,0.2295,0.7572,0.9157,1.0


interaction_count         lead_score          0.9157
number_of_courses_viewed  lead_score          0.7572
                          interaction_count   0.7216
annual_income             interaction_count   0.1228
Length: 4, dtype: float64

### Split the data

- Split the data with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

- Make sure that the target value `converted` is not in your dataframe.

In [7]:
X = df.drop(columns=TARGET_VAR)
y = df[TARGET_VAR].to_numpy()
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)


### Question 3

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source` ✅ 
- `employment_status`

In [8]:
X_train.select_dtypes('string').apply(mutual_info_score, args=(y_train,)).sort_values(ascending=False).round(2)


lead_source         0.03
employment_status   0.02
industry             0.0
location             0.0
Length: 4, dtype: float64

### Question 4

- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65 ✅ 
- 0.75
- 0.85

In [9]:
def train_logistic_reg(X_train, y_train, X_val, y_val, feature: str | None = None, C: float = 1.0):
    if feature is not None:
        X_train = X_train.drop(columns=feature)
        X_val = X_val.drop(columns=feature)

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(X_train.to_dict(orient='records'))
    X_val = dv.transform(X_val.to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    return model.score(X_val, y_val)


accuracy = train_logistic_reg(X_train, y_train, X_val, y_val)
round(accuracy, 2)


0.65

### Question 5

- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- `'lead_source'` ✅ 
- `'number_of_courses_viewed'`
- `'interaction_count'`

> **Note**: The difference doesn't have to be positive.

In [10]:
result = []
for feature in X_train.columns:
    accuracy_wo_feature = train_logistic_reg(X_train, y_train, X_val, y_val, feature=feature)
    result.append((
        feature,
        accuracy_wo_feature,
        accuracy - accuracy_wo_feature,
    ))


pd.DataFrame(
    result,
    columns=['feature_eliminated', 'accuracy_wo_feature', 'accuracy_diff'],
).sort_values('accuracy_diff', key=abs)


,feature_eliminated,accuracy_wo_feature,accuracy_diff
1,industry,0.645,0.0
2,employment_status,0.645,0.0
3,location,0.645,0.0
7,lead_score,0.644,0.0010000000000000009
5,number_of_courses_viewed,0.643,0.0020000000000000018
0,lead_source,0.642,0.0030000000000000027
6,interaction_count,0.601,0.04400000000000004
4,annual_income,0.724,-0.07899999999999996


### Question 6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001 ✅ 

> **Note**: If there are multiple options, select the smallest `C`.

In [11]:
accuracies = []
for C in [1e-6, 1e-5, 1e-4, 1e-3]:
    accuracy = train_logistic_reg(X_train, y_train, X_val, y_val, C=C)
    accuracies.append((
        C,
        accuracy,
        round(accuracy, 3),
    ))

pd.DataFrame(accuracies, columns=['C', 'accuracy', 'accuracy_round3']).sort_values('accuracy', ascending=False)


,C,accuracy,accuracy_round3
3,0.001,0.645,0.645
2,0.0001,0.613,0.613
0,1e-06,0.598,0.598
1,1e-05,0.598,0.598
